# Non-Instruction Fine-Tuning for HR Assistant

This notebook performs domain adaptation using raw HR policy text.
The goal is to adapt the base model to HR domain language, terminology, and writing style before instruction tuning.

In [45]:
# Install required packages
%pip install -q unsloth transformers datasets accelerate peft trl bitsandbytes

# For Colab, you may need to restart runtime after this cell


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [46]:
import torch
from unsloth import FastLanguageModel
from datasets import Dataset
import json

# Check GPU availability
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU available: False


In [47]:
# Load raw domain text
with open('../data/non_instruction_data.txt', 'r') as f:
    raw_text = f.read()

# Split into paragraphs
paragraphs = [p.strip() for p in raw_text.split('Paragraph ') if p.strip()]
print(f"Loaded {len(paragraphs)} paragraphs")

# Show sample
for i, p in enumerate(paragraphs[:3]):
    print(f"Paragraph {i+1}: {p[:200]}...")
    print()

Loaded 60 paragraphs
Paragraph 1: 1:
The company policy is reviewed annually and updated as necessary to ensure ongoing compliance with applicable laws and best practices....

Paragraph 2: 2:
Yes, employees will be notified of any significant changes to the company policy....

Paragraph 3: 3:
The company reviews its policy annually to ensure it remains compliant with current laws and reflects best practices in the industry....



In [48]:
# Chunk text for training
def chunk_text(text, max_length=512, overlap=50):
    """Split text into overlapping chunks."""
    words = text.split()
    chunks = []
    for i in range(0, len(words), max_length - overlap):
        chunk = ' '.join(words[i:i + max_length])
        if len(chunk.split()) > 50:  # Minimum chunk size
            chunks.append(chunk)
    return chunks

# Combine all paragraphs and chunk
full_text = ' '.join(paragraphs)
chunks = chunk_text(full_text, max_length=512, overlap=50)
print(f"Created {len(chunks)} chunks")

# Create dataset
dataset = Dataset.from_dict({"text": chunks})

Created 5 chunks


In [49]:
# Load base model with Unsloth
max_seq_length = 512
dtype = None  # Auto detection
load_in_4bit = True  # Use QLoRA

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-0.5B",  # or "unsloth/Llama-3.2-1B"
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 79137.81it/s]


Unsloth: Loading unsloth/Qwen2.5-0.5B via mlx-lm (runtime 4-bit affine quantization)...


Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 199728.76it/s]


[INFO] Quantized model with 7.671 bits per weight.
Unsloth: Quantized text model to 4-bit affine.


In [50]:
# Apply LoRA/QLoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=16,           # Rank
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,  # Alpha
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

Unsloth: LoRA applied — 8,798,208 trainable params (4.38% of 200,914,816 total)


In [51]:
# 1. Force the model object to expose its internal config structure
if not hasattr(model, "config") and hasattr(model, "model") and hasattr(model.model, "config"):
    model.config = model.model.config

# 2. Check if SFTTrainer can read it now
print("Has config attribute now?:", hasattr(model, "config"))


Has config attribute now?: False


In [53]:
# Format dataset for training

def formatting_prompts_func(examples):
    return {"text": examples["text"]}

# Apply formatting to the single Dataset object
# This is not a DatasetDict, so we split directly.
dataset = dataset.map(formatting_prompts_func, batched=True)

# Split the single Dataset into train/test
split_dataset = dataset.train_test_split(test_size=0.1, seed=42)

# Assign your final clean variables
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print(f"Train samples: {len(train_dataset)}")
print(f"Eval samples: {len(eval_dataset)}")

'''
def formatting_prompts_func(examples):
    return {"text": examples["text"]}

dataset = dataset.map(formatting_prompts_func, batched=True)

# Split train/eval
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

print(f"Train samples: {len(train_dataset)}")
print(f"Eval samples: {len(eval_dataset)}")
'''

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Map: 100%|██████████| 5/5 [00:00<00:00, 108.97 examples/s]

Train samples: 4
Eval samples: 1


'\ndef formatting_prompts_func(examples):\n    return {"text": examples["text"]}\n\ndataset = dataset.map(formatting_prompts_func, batched=True)\n\n# Split train/eval\ndataset = dataset.train_test_split(test_size=0.1, seed=42)\ntrain_dataset = dataset["train"]\neval_dataset = dataset["test"]\n\nprint(f"Train samples: {len(train_dataset)}")\nprint(f"Eval samples: {len(eval_dataset)}")\n'

In [71]:
import numpy as np
import mlx.core as mx

def _sample_next_token(logits, temperature=1.0, top_p=0.9):
    if temperature == 0.0:
        return int(np.argmax(logits))

    scores = logits.astype(np.float64) / temperature
    scores -= np.max(scores)
    exp_scores = np.exp(scores)
    probs = exp_scores / np.sum(exp_scores)

    sorted_indices = np.argsort(-probs)
    cumulative_probs = np.cumsum(probs[sorted_indices])
    selected = sorted_indices[cumulative_probs <= top_p]
    if selected.size == 0:
        selected = sorted_indices[:1]

    top_probs = probs[selected]
    top_probs /= top_probs.sum()
    return int(np.random.choice(selected, p=top_probs))

def _generate_text(model, tokenizer, prompt, max_new_tokens=50, temperature=0.7, top_p=0.9):
    inputs = tokenizer._tokenizer(prompt, return_tensors="pt")
    token_ids = inputs["input_ids"][0].cpu().numpy().tolist()

    for _ in range(max_new_tokens):
        tokens = mx.array(np.array([token_ids], dtype=np.int64))
        logits = np.asarray(model(tokens)[0, -1])
        next_token = _sample_next_token(logits, temperature=temperature, top_p=top_p)
        token_ids.append(next_token)
        if next_token == tokenizer.eos_token_id:
            break

    return tokenizer._tokenizer.decode(token_ids, skip_special_tokens=True)

In [72]:
# Training arguments
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments

training_args = SFTConfig(
    output_dir="../models/non_instruction_ft",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    max_steps=100,  # Small experiment
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=20,
    save_steps=50,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=3407,
    report_to="none",
    dataset_text_field="text",
)

In [55]:
# 1. Drill past the Unsloth wrapper layers to find the actual Hugging Face config
if not hasattr(model, "config"):
    for inner_attr in ["model", "base_model", "patcher"]:
        if hasattr(model, inner_attr):
            inner_obj = getattr(model, inner_attr)
            # Check if the nested object or its underlying base has the config
            if hasattr(inner_obj, "config"):
                model.config = inner_obj.config
                break
            elif hasattr(inner_obj, "base_model") and hasattr(inner_obj.base_model, "config"):
                model.config = inner_obj.base_model.config
                break

# 2. Fallback: If it's still missing, grab the configuration directly from the tokenizer module
if not hasattr(model, "config") and hasattr(tokenizer, "model_max_length"):
    from transformers import AutoConfig
    print("orcing a manual configuration build for Qwen 0.5B...")
    model.config = AutoConfig.from_pretrained("unsloth/Qwen2.5-0.5B")

# 3. Final Verification check
print("Model Config Fixed Successfully?:", hasattr(model, "config"))


orcing a manual configuration build for Qwen 0.5B...
Model Config Fixed Successfully?: True


In [60]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir="../models/non_instruction_ft",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    eval_strategy="steps",
    dataset_text_field="text",
    # max_seq_length=2048,
    
    # 👑 FIXES FOR APPLE SILICON:
    optim="adamw_torch",           #  Bypasses CUDA-only bitsandbytes
    # If using standard Hugging Face AutoModel (previous step workaround):
    # use_cpu=True                 # Prevents deep CUDA device check conflicts
)


In [73]:
from pathlib import Path

output_dir = Path("../models/non_instruction_ft_adapter")
output_dir.mkdir(parents=True, exist_ok=True)

# Save the adapter/model weights using the qwen2 API.
# `save_weights` writes the model parameters to a weights file.
model.save_weights(str(output_dir / "model_weights.safetensors"))

# Save the tokenizer using Hugging Face-compatible export if available.
if hasattr(tokenizer, "save_pretrained"):
    tokenizer.save_pretrained(output_dir)
elif hasattr(tokenizer, "save"):
    tokenizer.save(str(output_dir / "tokenizer.bin"))
else:
    print("Tokenizer export is unavailable. Save the tokenizer manually if needed.")

# Also save merged model (optional)
# model.save_pretrained_merged("../../models/non_instruction_ft_merged", tokenizer, save_method="merged_16bit")

In [74]:
# Test the model after non-instruction fine-tuning
FastLanguageModel.for_inference(model)

test_prompts = [
    "The company policy is reviewed",
    "Employees will be notified of",
    "Work from home policy states",
    "Leave requests must be approved by",
    "The compliance team is responsible for"
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running inference on device: {device}")

for prompt in test_prompts:
    prompt_text = f"### Instruction:\n{prompt}\n\n### Response:\n"
    generated = _generate_text(
        model,
        tokenizer,
        prompt_text,
        max_new_tokens=50,
        temperature=0.7,
        top_p=0.9,
    )
    print(f"Prompt: {prompt}")
    print(f"Generated: {generated}")
    print("---")

Running inference on device: cpu
Prompt: The company policy is reviewed
Generated: ### Instruction:
The company policy is reviewed

### Response:
The company policy is reviewed on a yearly basis and in line with the company policy and guidelines.

### Instruction:
The company policy is reviewed

### Response:
The company policy is reviewed on a yearly basis and in line with the company policy and guidelines.


---
Prompt: Employees will be notified of
Generated: ### Instruction:
Employees will be notified of

### Response:
### For this task, you will have to create a class in Python to represent an employee. You will have to define the following attributes:

### Name
### Job Title
### Age
### Salary
### Responsibilities
### Skills

### Instructions:

---
Prompt: Work from home policy states
Generated: ### Instruction:
Work from home policy states

### Response:
1. **Write a program that defines a function called `is_vacation`**.
2. **Inside `is_vacation`** function, **call the `is_day`